# 로컬 AI 9강 · 로컬 AI와 Uncensored 이미지 생성
AI CITY BUILDERS · AI BUSINESS UNIVERSITY

**ComfyUI 없이, 셀을 순서대로 실행합니다.**

- `light`: SD-Turbo FP16. 작은 입문용 모델로 흐름을 먼저 이해합니다. Uncensored 모델은 아닙니다.
- `qwen`: 사용자가 본 Qwen-Image-2.1-Uncensored-GGUF Q4 + Qwen3-VL-8B Q4 인코더 + 전용 VAE. 세 파일 약 10.31 GB. 충분한 RAM이 있는 GPU 런타임용 선택 실습입니다.
- **Qwen 경로는 Colab GPU 전체 생성 미검증**입니다. 공식 엔진의 지원 문서와 옵션을 바탕으로 구성했으며, 무료 Colab 실행을 보장하지 않습니다.
- 다운로드 전 해당 모델 카드의 라이선스를 확인하세요. 무료 Colab에서 외부 웹 UI를 띄우지 않고 노트북 안에서 실험합니다.
- LLM 실습과 **별도 새 런타임**에서 시작하세요. 1번에서 실제 자원을 확인하고 부족하면 큰 파일을 받기 전에 중단합니다.

[셀별 비주얼 해설](https://www.aicitybuilders.com/la8b) · [SD-Turbo 모델 카드](https://huggingface.co/stabilityai/sd-turbo) · [Qwen GGUF 모델 카드](https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF) · [공식 엔진 사용법](https://github.com/leejet/stable-diffusion.cpp/blob/a1ded76da5818803fca97a3b433669ef727d32cf/docs/qwen_image_2.1.md) · [Colab 자원 안내](https://research.google.com/colaboratory/faq.html)


## 01. 먼저, 내 코랩 자원 확인

GPU 메모리, 시스템 메모리, 디스크는 서로 다른 공간입니다. 런타임 → 런타임 유형 변경에서 GPU를 선택하고 새 노트북에서 시작하세요.

**확인:** 무료·유료 모두 GPU 종류와 가용성은 보장되지 않습니다. 이 노트북은 코랩 GPU에서 아직 전체 생성 검증을 완료하지 않았습니다.


In [ ]:
import os, json, shutil, subprocess, sys, time
from pathlib import Path
import torch
GiB = 1024 ** 3
ram_total = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / GiB
disk_free = shutil.disk_usage(".").free / GiB
gpu_free = torch.cuda.mem_get_info()[0] / GiB if torch.cuda.is_available() else 0
print(f"시스템 RAM {ram_total:.1f} GiB / 디스크 여유 {disk_free:.1f} GiB / GPU 여유 {gpu_free:.1f} GiB")
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
else: print("GPU가 없습니다. 런타임 유형을 GPU로 바꾼 뒤 실행하세요.")

## 02. 가벼운 시작과 Qwen 실험 선택

처음에는 light로 한 장을 만들어 봅니다. qwen은 앞서 본 Qwen-Image-2.1 GGUF를 실행하는 선택 과정입니다. 두 모델은 성능·용도·라이선스가 다릅니다.

**확인:** SD-Turbo는 빠른 입문용 이미지 생성 모델입니다. uncensored 모델이나 Qwen의 축소판이라고 소개하지 않습니다.


In [ ]:
MODE = "light"  #@param ["light", "qwen"]
SEED = 8  #@param {type:"integer"}
PROMPT = "An elegant miniature university campus, terracotta buildings, teal glass, warm ivory background, architectural model photography, no text"  #@param {type:"string"}
WIDTH = 512
HEIGHT = 512
if not torch.cuda.is_available(): raise RuntimeError("GPU 런타임으로 바꾼 뒤 1번부터 실행하세요.")
if MODE == "qwen":
    print("Q4 이미지 모델 + Q4 텍스트 인코더 + VAE: 약 10.31 GB. 실행 메모리는 별도입니다.")
    if ram_total < 22 or gpu_free < 10 or disk_free < 25:
        raise RuntimeError("수업의 보수적 시작 기준: RAM 22 GiB, GPU 여유 10 GiB, 디스크 여유 25 GiB 이상. 고용량 런타임 또는 light를 선택하세요. 이는 공식 최소 사양이나 실행 보장이 아닙니다.")
elif gpu_free < 4 or disk_free < 8:
    raise RuntimeError("다른 모델을 종료하고 GPU 여유 4 GiB, 디스크 여유 8 GiB 이상에서 시작하세요. 실습 기준이며 실행 보장은 아닙니다.")
print("선택:", MODE, "· 한 장 ·", WIDTH, "×", HEIGHT)

## 03. 실행 도구 설치

light는 Python Diffusers를 사용합니다. qwen은 공식 C++ 엔진을 CUDA로 빌드합니다. ComfyUI나 외부 웹 UI를 띄우지 않고 셀 안에서 실행합니다.

**확인:** C++ 빌드에는 시간이 걸립니다. 설치 후 패키지 충돌 안내가 나오면 런타임을 다시 시작하고 1번부터 실행하세요.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "huggingface_hub==0.36.0", "Pillow==11.3.0"], check=True)
ENGINE_REV = "a1ded76da5818803fca97a3b433669ef727d32cf"
if MODE == "light":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "diffusers==0.35.1", "transformers==4.57.6", "accelerate==1.10.1"], check=True)
else:
    if not shutil.which("nvcc"): raise RuntimeError("CUDA 빌드 도구가 있는 GPU 런타임이 필요합니다.")
    if not shutil.which("cmake"): subprocess.run([sys.executable,"-m","pip","install","-q","cmake==3.31.6"],check=True)
    ENGINE = Path("/content/aicb-sd-cpp")
    if not ENGINE.exists(): subprocess.run(["git","clone","https://github.com/leejet/stable-diffusion.cpp",str(ENGINE)],check=True)
    subprocess.run(["git","-C",str(ENGINE),"checkout",ENGINE_REV],check=True)
    subprocess.run(["git","-C",str(ENGINE),"submodule","update","--init","--recursive"],check=True)
    subprocess.run(["cmake","-S",str(ENGINE),"-B",str(ENGINE/"build"),"-DSD_CUDA=ON","-DSD_WEBP=OFF","-DSD_WEBM=OFF","-DCMAKE_BUILD_TYPE=Release"],check=True)
    subprocess.run(["cmake","--build",str(ENGINE/"build"),"--config","Release","-j","2"],check=True)

## 04. 필요한 파일만 받기

Qwen은 저장소 전체가 아니라 필요한 세 파일만 받습니다. 텍스트 생성에는 이미지 편집용 mmproj 파일을 받지 않습니다.

**확인:** 코랩 VM이 종료되면 다시 받을 수 있습니다. GB는 10억 바이트, GiB는 2의 30승 바이트입니다. 파일 용량과 RAM·VRAM 사용량은 다릅니다.


In [ ]:
from huggingface_hub import hf_hub_download
SD_REPO = "stabilityai/sd-turbo"
SD_REV = "b261bac6fd2cf515557d5d0707481eafa0485ec2"
IMAGE_REPO = "abenzerps/Qwen-Image-2.1-Uncensored-GGUF"
IMAGE_REV = "6b34e59458d3eb7ba6a6f86a116aed5253dc02c3"
ENC_REPO = "Qwen/Qwen3-VL-8B-Instruct-GGUF"
ENC_REV = "f982a07559d4a2f6c8744d840bf6fccab30eea96"
if MODE == "qwen":
    diffusion = hf_hub_download(IMAGE_REPO,"qwen-image-2.1-UC-Q4_K_M.gguf",revision=IMAGE_REV)
    encoder = hf_hub_download(ENC_REPO,"Qwen3VL-8B-Instruct-Q4_K_M.gguf",revision=ENC_REV)
    vae = hf_hub_download(IMAGE_REPO,"vae/qwen_image_2.1_vae_bf16.safetensors",revision=IMAGE_REV)
    print("세 파일 합계:", round(sum(Path(p).stat().st_size for p in [diffusion,encoder,vae])/1e9,2),"GB")
else:
    print("SD-Turbo FP16 가중치 약 2.58 GB + 설정·토크나이저. 다음 셀에서 필요한 구성요소를 받습니다.")

## 05. 모델과 설정 연결

같은 seed를 기록하면 비교하기 쉽습니다. Qwen은 CPU offload와 Flash Attention을 사용해 GPU 부담을 줄입니다. 대신 시스템 메모리와 전송 비용이 생깁니다.

**확인:** CPU offload는 메모리를 없애는 기능이 아닙니다. GPU와 RAM 사이에 옮기는 방식입니다. T4에서는 FP16을 사용하며 BF16을 전제로 하지 않습니다.


In [ ]:
if MODE == "light":
    from diffusers import AutoPipelineForText2Image
    pipe = AutoPipelineForText2Image.from_pretrained(SD_REPO,revision=SD_REV,
        torch_dtype=torch.float16,variant="fp16",use_safetensors=True)
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_tiling()
    print("SD-Turbo: 512px, 1 step, guidance 0으로 시작합니다.")
else:
    CLI = ENGINE / "build/bin/sd-cli"
    help_text = subprocess.run([str(CLI),"--help"],capture_output=True,text=True,check=True).stdout
    for option in ["--diffusion-model","--llm","--vae","--offload-to-cpu","--fa"]:
        if option not in help_text: raise RuntimeError("엔진 옵션 확인 필요: " + option)
    command = [str(CLI),"--diffusion-model",diffusion,"--llm",encoder,"--vae",vae,
        "-p",PROMPT,"--cfg-scale","6.0","--sampling-method","euler",
        "--steps","20","-W",str(WIDTH),"-H",str(HEIGHT),"-s",str(SEED),
        "--offload-to-cpu","--fa","-o","aicb-image.png"]
    print("Qwen Image: 512px, 20 steps. 아직 생성 전입니다.")

## 06. 한 장 생성하고 화면에서 보기

같은 문장을 숫자 표현으로 바꾼 뒤, 이미지 표현을 만들고 픽셀로 복원합니다. 옆 그래픽은 개념도이며 아래 실행 결과만 실제 모델 출력입니다.

**확인:** 메모리 부족이면 새 런타임에서 한 모델만 실행하세요. Qwen은 더 큰 RAM/GPU를 선택하거나 light로 돌아가세요. 해상도·배치 크기도 메모리에 영향을 줍니다.


In [ ]:
started = time.time()
if MODE == "light":
    image = pipe(prompt=PROMPT,num_inference_steps=1,guidance_scale=0.0,
        width=WIDTH,height=HEIGHT,generator=torch.Generator("cpu").manual_seed(SEED)).images[0]
    image.save("aicb-image.png")
else:
    with open("aicb-image-engine.log","w") as log:
        result = subprocess.run(command,stdout=log,stderr=subprocess.STDOUT)
    if result.returncode:
        print(Path("aicb-image-engine.log").read_text()[-5000:])
        raise RuntimeError("생성이 끝나지 않았습니다. 아래 메모리 안내와 로그를 확인하세요.")
    from PIL import Image
    image = Image.open("aicb-image.png")
elapsed = round(time.time()-started,1)
from IPython.display import display
display(image)
print("실제 소요 시간:",elapsed,"초")

## 07. 프롬프트와 조건을 함께 저장

그림만 남기지 말고 모델 버전, 프롬프트, seed와 실행 조건을 같이 남깁니다. 다른 GPU·엔진에서는 같은 seed라도 픽셀이 달라질 수 있습니다.

**확인:** 이미지 속 글자·손·구조가 정확한지도 직접 확인합니다. 생성 성공과 결과 품질은 별개의 판단입니다.


In [ ]:
record = {"mode":MODE,"prompt":PROMPT,"seed":SEED,"width":WIDTH,"height":HEIGHT,
    "elapsed_seconds":elapsed,"gpu":torch.cuda.get_device_name(0),"torch":torch.__version__,
    "model":SD_REPO if MODE=="light" else IMAGE_REPO,
    "revision":SD_REV if MODE=="light" else IMAGE_REV}
if MODE=="qwen": record.update(encoder=ENC_REPO,encoder_revision=ENC_REV,engine_revision=ENGINE_REV)
else: record.update(diffusers=__import__('diffusers').__version__)
Path("aicb-image-settings.json").write_text(json.dumps(record,ensure_ascii=False,indent=2))
print("왼쪽 파일 목록에서 aicb-image.png와 aicb-image-settings.json을 내려받으세요.")